# VisionScribe AI: train and evaluate on Google Colab

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. It takes about 1–1.5 hours and trains a DenseNet-121 model and a ResNet-50 baseline, then prints test-set BLEU scores for both.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/khush-ahuja/VisionScribe-AI.git
%cd VisionScribe-AI
!pip -q install -r requirements.txt

## Download Flickr8k (8,091 images, 5 captions each)

In [ ]:
!wget -q https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip
!wget -q https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip
!unzip -q -o Flickr8k_Dataset.zip && unzip -q -o Flickr8k_text.zip

In [ ]:
import csv, os, shutil
from pathlib import Path
root = Path("data/Flickr8k"); img_dir = root / "Images"; img_dir.mkdir(parents=True, exist_ok=True)
src = Path("Flicker8k_Dataset")
for f in src.glob("*.jpg"):
    shutil.move(str(f), img_dir / f.name)
have = {f.name for f in img_dir.glob("*.jpg")}
rows = []
for line in open("Flickr8k.token.txt", encoding="utf-8"):
    if "\t" not in line:
        continue
    key, cap = line.rstrip("\n").split("\t", 1)
    img = key.split("#")[0]
    if img in have:
        rows.append((img, cap.strip()))
with open(root / "captions.txt", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["image", "caption"]); w.writerows(rows)
print(f"{len(have)} images, {len(rows)} captions")


## Train both encoders

In [ ]:
!python train.py --data data/Flickr8k --backbone densenet121 --out runs/densenet121 --epochs 12 --workers 2

In [ ]:
!python train.py --data data/Flickr8k --backbone resnet50 --out runs/resnet50 --epochs 12 --workers 2

## Test-set BLEU: DenseNet-121 vs ResNet-50 baseline

In [ ]:
!python evaluate.py --data data/Flickr8k --runs runs/densenet121 runs/resnet50 --beam 3
!cat results/test_bleu.json

## Try it on a few test images

In [ ]:
import random
from IPython.display import Image, display
from PIL import Image as PILImage
from evaluate import load_run
from caption import caption_image
from visionscribe.data import load_captions, split_images

model, vocab = load_run('runs/densenet121', 'cuda')
caps = load_captions('data/Flickr8k/captions.txt')
_, _, test = split_images(caps.keys())
for name in random.Random(0).sample(test, 4):
    path = f'data/Flickr8k/Images/{name}'
    display(Image(path, width=320))
    print('VisionScribe:', caption_image(model, vocab, PILImage.open(path), 'cuda'))
    print('Human:      ', caps[name][0], '\n')